random forest getting outperformed by catboost,using catboost for noiw as its way similar

In [ ]:
import pandas as pd
import numpy as np
!pip install scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

print("=== PREPARING DATA FOR MODELING ===\n")

# Define features
categorical_features = ['service_name', 'comm_plan_name', 'case_origin']
numerical_features = [
    'council_district', 'year', 'month', 'day_of_week', 'hour',
    'is_infrastructure', 'is_followup', 'description_length'
]

all_features = categorical_features + numerical_features

# Prepare X and y
X = model_data[all_features].copy()
y = model_data['log_wait'].copy()

print(f"Total samples: {len(X):,}")
print(f"Features: {len(all_features)}")
print(f"  - Categorical: {len(categorical_features)}")
print(f"  - Numerical: {len(numerical_features)}")

# Check for any missing values
print(f"\nMissing values:")
print(X.isnull().sum()[X.isnull().sum() > 0])

# Time-based train/test split (oldest 80% train, newest 20% test)
sorted_idx = model_data['date_requested'].sort_values().index
split_point = int(len(sorted_idx) * 0.8)
train_idx = sorted_idx[:split_point]
test_idx = sorted_idx[split_point:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"\n✓ Train set: {len(X_train):,} samples")
print(f"✓ Test set: {len(X_test):,} samples")
print(f"Train date range: {model_data.loc[train_idx, 'date_requested'].min()} to {model_data.loc[train_idx, 'date_requested'].max()}")
print(f"Test date range: {model_data.loc[test_idx, 'date_requested'].min()} to {model_data.loc[test_idx, 'date_requested'].max()}")

# ==========================================
# BUILD MODEL PIPELINE
# ==========================================

print("\n=== BUILDING MODEL ===\n")

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    verbose_feature_names_out=False
)

# Full pipeline with Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,      # Number of trees
        max_depth=20,          # Prevent overfitting
        min_samples_split=100, # Don't split nodes with <100 samples
        min_samples_leaf=50,   # Each leaf needs >=50 samples
        random_state=42,
        n_jobs=-1,             # Use all CPU cores
        verbose=1              # Show progress
    ))
])

# Train model
print("Training Random Forest...")
print("(This may take a few minutes...)\n")
rf_pipeline.fit(X_train, y_train)
print("\n✓ Model trained successfully!")

# ==========================================
# EVALUATE MODEL
# ==========================================

print("\n=== MODEL EVALUATION ===\n")

# Make predictions
y_train_pred = rf_pipeline.predict(X_train)
y_test_pred = rf_pipeline.predict(X_test)

# Calculate metrics (log-space + original day-space)
def evaluate_model(y_true_log, y_pred_log, dataset_name):
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2_log = r2_score(y_true_log, y_pred_log)

    y_true_days = np.expm1(y_true_log)
    y_pred_days = np.expm1(y_pred_log)

    mae_days = mean_absolute_error(y_true_days, y_pred_days)
    rmse_days = np.sqrt(mean_squared_error(y_true_days, y_pred_days))

    print(f"{dataset_name}:")
    print(f"  Log-space MAE:  {mae_log:.3f}")
    print(f"  Log-space RMSE: {rmse_log:.3f}")
    print(f"  Log-space R²:   {r2_log:.3f}")
    print(f"  Day-space MAE:  {mae_days:.2f} days")
    print(f"  Day-space RMSE: {rmse_days:.2f} days")
    print()

    return mae_log, rmse_log, r2_log, mae_days, rmse_days

train_mae_log, train_rmse_log, train_r2, train_mae_days, train_rmse_days = evaluate_model(y_train, y_train_pred, "Training Set")
test_mae_log, test_rmse_log, test_r2, test_mae_days, test_rmse_days = evaluate_model(y_test, y_test_pred, "Test Set")

# Check for overfitting
if train_r2 - test_r2 > 0.1:
    print("⚠️  Warning: Model may be overfitting (train R² >> test R²)")
else:
    print("✓ Model generalizes well (no significant overfitting)")

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

print("\n=== FEATURE IMPORTANCE ===\n")

# Get feature names after one-hot encoding
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Get importance scores
importances = rf_pipeline.named_steps['regressor'].feature_importances_

# Create DataFrame
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance_df.head(15).to_string(index=False))

# ==========================================
# VISUALIZATIONS
# ==========================================

print("\n=== CREATING VISUALIZATIONS ===\n")

# 1. Actual vs Predicted (converted back to days)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

y_train_days = np.expm1(y_train)
y_train_pred_days = np.expm1(y_train_pred)
y_test_days = np.expm1(y_test)
y_test_pred_days = np.expm1(y_test_pred)

max_day = np.percentile(np.concatenate([y_train_days.values, y_test_days.values]), 99)

# Training set
axes[0].scatter(y_train_days, y_train_pred_days, alpha=0.3, s=1)
axes[0].plot([0, max_day], [0, max_day], 'r--', lw=2)
axes[0].set_xlabel('Actual Response Time (days)')
axes[0].set_ylabel('Predicted Response Time (days)')
axes[0].set_title(f'Training Set\nLog-R² = {train_r2:.3f}, MAE = {train_mae_days:.1f} days')
axes[0].set_xlim(0, max_day)
axes[0].set_ylim(0, max_day)

# Test set
axes[1].scatter(y_test_days, y_test_pred_days, alpha=0.3, s=1)
axes[1].plot([0, max_day], [0, max_day], 'r--', lw=2)
axes[1].set_xlabel('Actual Response Time (days)')
axes[1].set_ylabel('Predicted Response Time (days)')
axes[1].set_title(f'Test Set\nLog-R² = {test_r2:.3f}, MAE = {test_mae_days:.1f} days')
axes[1].set_xlim(0, max_day)
axes[1].set_ylim(0, max_day)

plt.tight_layout()
plt.savefig('model_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: model_predictions.png")

# 2. Feature Importance (Top 20)
plt.figure(figsize=(10, 8))
top_20 = feature_importance_df.head(20)
plt.barh(range(len(top_20)), top_20['importance'])
plt.yticks(range(len(top_20)), top_20['feature'])
plt.xlabel('Importance Score')
plt.title('Top 20 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: feature_importance.png")

# 3. Residuals
residuals_test = y_test_days - y_test_pred_days

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(residuals_test, bins=50, edgecolor='black')
plt.xlabel('Residuals (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Errors')
plt.axvline(0, color='red', linestyle='--', linewidth=2)

plt.subplot(1, 2, 2)
plt.scatter(y_test_pred_days, residuals_test, alpha=0.3, s=1)
plt.xlabel('Predicted Response Time (days)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residual Plot')
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlim(0, max_day)

plt.tight_layout()
plt.savefig('residuals.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: residuals.png")

# ==========================================
# SAVE MODEL
# ==========================================

import joblib

joblib.dump(rf_pipeline, 'rf_model.pkl')
print("\n✓ Model saved as 'rf_model.pkl'")

# ==========================================
# SUMMARY
# ==========================================

print("\n" + "="*50)
print("MODEL TRAINING COMPLETE!")
print("="*50)
print(f"\n📊 Performance Summary:")
print(f"   Test MAE: {test_mae_days:.2f} days (avg prediction error)")
print(f"   Test RMSE: {test_rmse_days:.2f} days")
print(f"   Test Log-R²: {test_r2:.3f} (explains {test_r2*100:.1f}% of log-variance)")
print(f"\n📁 Files Created:")
print(f"   - rf_model.pkl (trained model)")
print(f"   - model_predictions.png")
print(f"   - feature_importance.png")
print(f"   - residuals.png")
print(f"\n🎯 Next Steps:")
print(f"   1. Analyze feature importance results")
print(f"   2. Investigate largest prediction errors")
print(f"   3. Try other models (XGBoost, LightGBM)")
print(f"   4. Add to portfolio!")
